# Ngôn Sắc Việt — Train NLLB + LangID trên Kaggle CPU 12h (Không phải xếp hàng GPU)

Notebook này chạy trực tiếp trên **Kaggle CPU (0 queue, tối đa 12 giờ chạy nền dù tắt máy tính)** với cấu hình gọn + tự động ngắt ở mốc **8.5 giờ** (`MAX_TRAIN_SECONDS = 30600`) để dành 3.5 giờ chấm điểm `evaluate.py` (`--beams 1`), smoke test và đóng gói `/kaggle/working/nllb-ethnic-ready.zip` + `/kaggle/working/langid.json`.

In [ ]:
REPO_URL = "https://github.com/minhduy6868/vietlangai-model.git"
BRANCH = "main"
BASE_MODEL = "facebook/nllb-200-distilled-600M"
RANK = 32
EPOCHS = 1
LR = 3e-4
BATCH = 16
GRAD_ACCUM = 4
MAX_LENGTH = 128
MAX_PER_LANG = 4000
CONCAT_RATIO = 0.2
LONG_UPSAMPLE = 1
SAVE_STEPS = 50
LOGGING_STEPS = 10
MAX_TRAIN_SECONDS = 30600


In [ ]:
import os
import subprocess
import sys

import torch

print("CPUs:", os.cpu_count(), "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
with open("/proc/meminfo", encoding="utf-8") as f:
    for line in f:
        if line.startswith(("MemTotal:", "MemAvailable:")):
            print(line.strip())

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.56,<5", "peft>=0.17", "accelerate>=1.0",
    "sacrebleu>=2.4", "sentencepiece>=0.2", "pyarrow>=15",
])


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "vietlangai-model"

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Working dir:", Path.cwd(), "| Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

subprocess.check_call([sys.executable, "-u", "training/import_public.py"])
subprocess.check_call([sys.executable, "-u", "training/train_langid.py"])
langid_src = Path("models/langid.json")
if langid_src.is_file():
    shutil.copy2(langid_src, WORK / "langid.json")
    print("Đã xuất LangID:", WORK / "langid.json", (WORK / "langid.json").stat().st_size)


In [ ]:
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

sys.path.insert(0, "/kaggle/working/vietlangai-model")
from scripts.run_cloud_pipeline import package_bundle, run_smoke, run_streamed

OUT_DIR = Path("/kaggle/working/nllb-ethnic")
ZIP_OUT = Path("/kaggle/working/nllb-ethnic-ready.zip")
EVAL_OUT = Path("/kaggle/working/eval_report.json")

train_cmd = [
    sys.executable, "-u", "training/finetune_nllb.py",
    "--base", BASE_MODEL,
    "--output", str(OUT_DIR),
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--batch", str(BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--max-length", str(MAX_LENGTH),
    "--max-per-lang", str(MAX_PER_LANG),
    "--rank", str(RANK),
    "--concat-ratio", str(CONCAT_RATIO),
    "--long-upsample", str(LONG_UPSAMPLE),
    "--save-steps", str(SAVE_STEPS),
    "--logging-steps", str(LOGGING_STEPS),
    "--workers", "0",
    "--no-grad-checkpoint",
    "--resume",
]

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
print("Chạy CPU train (giới hạn thời gian", MAX_TRAIN_SECONDS, "giây):", " ".join(train_cmd), flush=True)
timed_out = False
proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)

def _watchdog():
    global timed_out
    deadline = time.monotonic() + MAX_TRAIN_SECONDS
    while proc.poll() is None:
        if time.monotonic() >= deadline:
            timed_out = True
            print(f"\nĐạt ngưỡng {MAX_TRAIN_SECONDS}s, dừng train êm để xuất & chấm điểm checkpoint...", flush=True)
            proc.terminate()
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                proc.kill()
            return
        time.sleep(10)

watcher = threading.Thread(target=_watchdog, daemon=True)
watcher.start()
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
watcher.join(timeout=5)

if rc == 0 and not timed_out:
    eval_cmd = [
        sys.executable, "-u", "training/evaluate.py",
        "--base", BASE_MODEL,
        "--adapter", str(OUT_DIR),
        "--split", "test",
        "--batch", "16",
        "--beams", "1",
        "--output", str(EVAL_OUT),
    ]
    run_streamed(eval_cmd)
    run_smoke(BASE_MODEL, OUT_DIR)
    package_bundle(OUT_DIR, ZIP_OUT, EVAL_OUT, None)
    print("[CPU_PIPELINE_COMPLETE]", flush=True)
else:
    ckpt_root = OUT_DIR / "checkpoints"
    saved = list(ckpt_root.glob("checkpoint-*")) if ckpt_root.is_dir() else []
    if not saved:
        raise RuntimeError(f"Train CPU kết thúc (rc={rc}, timed_out={timed_out}) khi chưa có checkpoint nào.")
    print(f"Chuyển sang export_checkpoint từ {len(saved)} checkpoint...", flush=True)
    run_streamed([
        sys.executable, "-u", "scripts/export_checkpoint.py",
        "--output", str(OUT_DIR),
        "--base", BASE_MODEL,
        "--languages", "rad", "tyz",
        "--method", f"lora-r{RANK}",
        "--evaluate",
        "--eval-beams", "1",
        "--eval-batch", "16",
        "--smoke",
        "--zip-out", str(ZIP_OUT),
        "--eval-out", str(EVAL_OUT),
    ])
    print("[CPU_EXPORT_COMPLETE]", flush=True)
